# Hybrid PyTorch & HeteroSymNN Pulsar Classification Pipeline

In this notebook, we implement a real-world scientific application: classifying candidate signals from the **High Time Resolution Universe Survey (HTRU 1)** to identify pulsars.

We build a hybrid computational pipeline:
1. **PyTorch Feature Extractor:** A deep CNN utilizing residual connections and Squeeze-and-Excitation (SE) attention to extract feature representations from 2D raw signal surfaces.
2. **HeteroSymNN Brain:** A customized symbolic classification layer using distinct activation functions (Fourier filters, generalist activation curves, etc.) representing physical priors.

We link the two frameworks together, backpropagating gradients from HeteroSymNN back through PyTorch, and explore the **Zero-Recompile Dynamic Tuning** capability to control search sensitivity.

In [ ]:
import sys
import os
import pickle
import urllib.request
import tarfile
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Setup path to import HeteroSymNN from the local directory
sys.path.insert(0, os.path.abspath('..'))

from HeteroSymNN.Core.Nets import MLP, HeteroLinearNet
from HeteroSymNN.Core import losses, optimizers, initializers
from HeteroSymNN.API import Wrapper
from HeteroSymNN.config import settings

# Enforce CPU JIT backend for HeteroSymNN in this demo
settings.set_default_compute_method("CPU_PYTHON")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch executing on: {device}")

## 1. Automated Dataset Management

We download the HTRU 1 dataset (174 MB compressed, 324 MB extracted) directly from the University of Manchester Jodrell Bank Centre for Astrophysics. We write a script to download and extract the data automatically.

In [ ]:
dataset_url = "http://www.jb.man.ac.uk/research/ascaife/htru1-batches-py.tar.gz"
tar_filename = "htru1-batches-py.tar.gz"
extract_dir = "htru1_data"

if not os.path.exists(tar_filename):
    print("Downloading HTRU 1 dataset (~174 MB)... This may take a minute.")
    urllib.request.urlretrieve(dataset_url, tar_filename)
    print("Download complete.")

if not os.path.exists(extract_dir):
    print("Extracting dataset...")
    with tarfile.open(tar_filename, "r:gz") as tar:
        tar.extractall(path=extract_dir)
    print("Extraction complete.")

DATA_DIR = os.path.join(extract_dir, "htru1-batches-py")
print(f"Dataset located in: {DATA_DIR}")

## 2. Defining the Data Loader

The dataset consists of flat candidate signal batches. We reconstruct them into 3-channel 32x32 image representations representing:
1. **Channel 0:** Integrated Profile (Phase vs Intensity)
2. **Channel 1:** DM-SNR Curve (Dispersion Measure search profile)
3. **Channel 2:** Time Profile (Phase vs Sub-integration)

In [ ]:
class SpinnDataset(Dataset):
    def __init__(self, data_dir, train=True):
        self.data = []
        self.labels = []
        
        file_names = [f"data_batch_{i}" for i in range(1, 6)] if train else ["test_batch"]
        
        for file_name in file_names:
            file_path = os.path.join(data_dir, file_name)
            if not os.path.exists(file_path):
                continue
                
            with open(file_path, 'rb') as fo:
                batch_dict = pickle.load(fo, encoding='latin1')
                self.data.append(batch_dict['data'])
                self.labels.extend(batch_dict['labels'])
        
        if self.data:
            # Reshape flat arrays to 3x32x32 spatial surfaces
            self.data = np.vstack(self.data).reshape(-1, 3, 32, 32).astype(np.float32)
            raw_labels = np.array(self.labels, dtype=np.float32)
            self.labels = 1.0 - raw_labels # Swap labels for binary alignment
        
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx]), torch.tensor(self.labels[idx])

train_dataset = SpinnDataset(DATA_DIR, train=True)
test_dataset = SpinnDataset(DATA_DIR, train=False)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"Loaded {len(train_dataset)} training samples and {len(test_dataset)} testing samples.")

## 3. Defining the Feature Extractor

We implement the PyTorch model which extracts features. It consists of CNN layers containing residual shortcuts (`ResBlock`) and attention blocks (`SEBlock`) to suppress noise and amplify pulsar channels.

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation (Channel Attention) Block."""
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ResBlock(nn.Module):
    """Standard Residual Shortcut block."""
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential( 
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return self.relu(out)

class SpinnFeatureExtractor(nn.Module):
    """Extracts a 128-dimensional feature embedding from input candidates."""
    def __init__(self, output_features=128):
        super(SpinnFeatureExtractor, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            ResBlock(32, 32),
            nn.MaxPool2d(2),
            ResBlock(32, 64),
            nn.MaxPool2d(2),
            ResBlock(64, 128),
            SEBlock(channels=128, reduction=16),
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128 * 4 * 4, output_features),
            nn.BatchNorm1d(output_features),
            nn.Tanh() # Restricts feature space to [-1, 1] for stable symbolic evaluation
        )
    def forward(self, x):
        return self.conv_layers(x)

## 4. The Hybrid Gradient Bridge

Since our model contains a PyTorch CNN and a HeteroSymNN JIT classifier, training requires passing gradients across framework boundaries:
1. **Forward:** PyTorch outputs feature embeddings -> converted to NumPy -> passed to HeteroSymNN JIT kernel.
2. **Backward:** HeteroSymNN JIT calculates error -> calculates the parameter gradient back to input features (`input_gradient`) -> converted back to PyTorch -> backward pass triggers through PyTorch parameters.

In [ ]:
def train_hybrid_epoch(cnn, cnn_optimizer, hetero_model, dataloader, device):
    cnn.train()
    total_loss = 0
    
    for batch_idx, (images, labels) in enumerate(dataloader):
        images = images.to(device) / 255.0
        
        # 1. PyTorch Forward Pass
        cnn_optimizer.zero_grad()
        features = cnn(images)
        features.retain_grad() # Keep gradients accessible for the bridge
        
        # 2. Convert PyTorch outputs to NumPy for HeteroSymNN JIT input
        numpy_features = features.detach().cpu().numpy().astype(np.float32)
        numpy_labels = labels.numpy().astype(np.float32)
        
        # Cast inputs based on compute backend (CPU JIT / GPU CUDA)
        numpy_features, numpy_labels = hetero_model.cast_arrays(numpy_features, numpy_labels)
        
        # 3. HeteroSymNN Forward, Loss & Parameter updates
        loss = hetero_model.train_step(numpy_features.T, numpy_labels)
        dL_dX_numpy = hetero_model.input_gradient
        
        # 4. Cast gradient back to PyTorch & Step PyTorch optimizer
        dL_dX_torch = torch.from_numpy(dL_dX_numpy.T).to(device)
        features.backward(gradient=dL_dX_torch)
        cnn_optimizer.step()
        
        total_loss += loss

    return total_loss / len(dataloader)

def extract_bridge_features(cnn_model, test_loader, device):
    """Extracts static test set embeddings to evaluate JIT network performance."""
    cnn_model.eval()
    all_features, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device) / 255.0
            features = cnn_model(images)
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())
            
    X_bridge = np.vstack(all_features).astype(np.float32)
    y_true = np.concatenate(all_labels).astype(np.float32)
    return X_bridge, y_true

## 5. Defining the Symbolic Activation Architecture

In the first layer (64 neurons), we define a heterogeneous bank of activations representing physical signals (Fourier banks like `sin(a*x)`, `cos(a*x)`, Lorentzian distributions, Swish activations, etc.).

In the final output node, we use a sigmoid scaled by a parameter `v` (`sigmoid(x)*v`), which acts as our sensitivity threshold parameter.

In [ ]:
cnn = SpinnFeatureExtractor(output_features=128).to(device)
cnn_optimizer = optim.Adam(cnn.parameters(), lr=0.001)

# Set up heterogeneous math formulas
exponents = [2, 3, 4, 5]
h_activations = [
    # Layer 0 (64 Neurons): Wide, varied basis functions
    [(f"x + tanh(x)**{exponents[i % len(exponents)]}", {}) for i in range(16)] +
    [("x / (1 + exp(-beta*x))", {"beta": (1+i)/8}) for i in range(16)] +
    [("sin(a*x)", {"a": (i+1)/8}) for i in range(8)] +
    [("cos(a*x)", {"a": (i+1)/8}) for i in range(8)] +
    [("1 / (1 + (x - b)**2)", {"b": (i - 8.0)/2.0}) for i in range(16)],
    
    # Layer 1 (16 Neurons): Council aggregator (with SiLU approximated via tanh)
    [("tanh(x)", {}) for _ in range(4)] +
    [("x * (1 + tanh(x / 2)) / 2", {}) for _ in range(4)] +
    [("x / (1 + Abs(x))", {}) for _ in range(4)] +
    [("exp(-x**2)", {}) for _ in range(4)],
    
    # Layer 2 (1 Neuron): Output with dynamic scaling threshold parameter 'v'
    [("sigmoid(x)*v", {"v": 1.0})]
]

initials_config = [initializers.LecunNormal(), initializers.LecunNormal(), initializers.XavierUniform()]
focal_loss_string = "-alpha * y_true * (1 - y_pred)**gamma * log(y_pred + 1e-7) - (1 - alpha) * (1 - y_true) * y_pred**gamma * log(1 - y_pred + 1e-7)"
custom_loss = losses.FlexibleLoss(focal_loss_string, {"alpha": 0.75, "gamma": 2.5})

hetero_model = HeteroLinearNet(
    num_inputs=128,
    detailed_activations=h_activations,
    initializer=initials_config,
    loss_function=custom_loss,
    optimizer=optimizers.AdamOptimizer(learning_rate=0.001)
)

## 6. Training the Hybrid Network

We train the model for 50 epochs, tracking loss convergence to visualize training stability.

In [ ]:
EPOCHS = 50
epoch_losses = []
print("Initiating Training...")

for epoch in range(EPOCHS):
    avg_loss = train_hybrid_epoch(cnn, cnn_optimizer, hetero_model, train_loader, device)
    epoch_losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d}/{EPOCHS:02d} | Avg Loss: {avg_loss:.4f}")

# Plotting training loss convergence
plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS + 1), epoch_losses, color='tab:blue', linewidth=2.5)
plt.title("Hybrid Training Loss Convergence", fontsize=14)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Flexible Focal Loss", fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

## 7. Zero-Recompile Parameter Evaluation

We extract test embeddings. Next, we use `change_constants()` to instantly adjust the final layer sensitivity dial parameter `v` between standard ($v=1.0$), strict ($v=1.75$), sensitive ($v=0.65$), and extreme ($v=4.5$).

We compile these outputs into a performance graph illustrating how shifting the symbolic threshold values dynamically changes accuracy, recall, and false positive rates.

In [ ]:
X_bridge, y_true = extract_bridge_features(cnn, test_loader, device)
eval_agent = Wrapper(hetero_model, work_type="class")

v_vals = [0.65, 1.0, 1.75, 4.5]
accuracies, recalls, fprs = [], [], []

for v in v_vals:
    # Mutate parameter dynamically
    hetero_model.change_constants({2: [(0, "v", v)]})
    metrics, counts = eval_agent.test_accuracy(X_bridge, y_true)
    
    accuracies.append(metrics.get('Accuracy', 0))
    tp, fn = counts['correc_pos'], counts['false_neg']
    fp, tn = counts['false_pos'], counts['correct_neg']
    recalls.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
    fprs.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)
    print(f"Threshold v={v:<5} | Confusion Matrix: {counts} | Accuracy: {accuracies[-1]:.4f}")

# Plotting threshold impact
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(v_vals))
ax.bar(x - 0.25, accuracies, 0.25, label='Accuracy', color='tab:blue')
ax.bar(x, recalls, 0.25, label='Recall (True Positive Rate)', color='tab:green')
ax.bar(x + 0.25, fprs, 0.25, label='False Positive Rate', color='tab:red')
ax.set_xticks(x)
ax.set_xticklabels([f"v={v}" for v in v_vals])
ax.set_title("Impact of Output Sensitivity Parameter 'v' on Performance Metrics")
ax.set_ylabel("Rate / Accuracy")
ax.grid(True, axis='y', ls='--', alpha=0.3)
ax.legend()
plt.show()

## 8. Astrophysics Interpretability

Because our activation functions are defined as explicit symbolic expressions, we can investigate which physical signals (like frequencies) are weighted highest. We extract these weights to map out the filter bank's energy profile, and plot activation states across model classes.

In [ ]:
def analyze_and_plot_fourier_frequencies(model):
    params = model.get_parameters()
    W1 = params['layer_0']['weights'] 
    W2 = params['layer_1']['weights'] 
    
    frequencies = [(i+1)/8 for i in range(8)]
    sine_indices = range(32, 40)
    cos_indices = range(40, 48)
    
    sin_incoming = [np.linalg.norm(W1[:, idx]) for idx in sine_indices]
    sin_outgoing = [np.mean(np.abs(W2[idx, :])) for idx in sine_indices]
    cos_incoming = [np.linalg.norm(W1[:, idx]) for idx in cos_indices]
    cos_outgoing = [np.mean(np.abs(W2[idx, :])) for idx in cos_indices]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    x_axis = np.arange(len(frequencies))
    
    # Sine Plot
    ax1.bar(x_axis - 0.2, sin_incoming, 0.4, label='Incoming Energy (Norm W1)', color='tab:purple')
    ax1.bar(x_axis + 0.2, sin_outgoing, 0.4, label='Outgoing Importance (Mean W2)', color='tab:pink')
    ax1.set_xticks(x_axis)
    ax1.set_xticklabels([f"{f:.3f}" for f in frequencies])
    ax1.set_title("Sine Filter Bank Weight Energies")
    ax1.set_xlabel("Frequency")
    ax1.legend()
    ax1.grid(True, axis='y', ls='--', alpha=0.3)
    
    # Cosine Plot
    ax2.bar(x_axis - 0.2, cos_incoming, 0.4, label='Incoming Energy (Norm W1)', color='tab:teal')
    ax2.bar(x_axis + 0.2, cos_outgoing, 0.4, label='Outgoing Importance (Mean W2)', color='tab:olive')
    ax2.set_xticks(x_axis)
    ax2.set_xticklabels([f"{f:.3f}" for f in frequencies])
    ax2.set_title("Cosine Filter Bank Weight Energies")
    ax2.set_xlabel("Frequency")
    ax2.legend()
    ax2.grid(True, axis='y', ls='--', alpha=0.3)
    
    plt.suptitle("Astro-Physics Spectral Energy Profiles", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

def calculate_and_plot_activation_statistics(X, y, model, agent):
    _ = agent.predict(X)
    A0 = model.layers[0].a.T
    A1 = model.layers[1].a.T
    
    A0_pulsar = A0[y == 0]
    A0_noise = A0[y == 1]
    A1_pulsar = A1[y == 0]
    A1_noise = A1[y == 1]
    
    print("\n--- Layer 0 Outputs (Pulsar vs. Noise Mean) ---")
    print(f"  Polynomial Hunters (0-15) | Pulsar Mean: {np.mean(A0_pulsar[:, 0:16]):.4f} | Noise Mean: {np.mean(A0_noise[:, 0:16]):.4f}")
    print(f"  Swish Generalists (16-31) | Pulsar Mean: {np.mean(A0_pulsar[:, 16:32]):.4f} | Noise Mean: {np.mean(A0_noise[:, 16:32]):.4f}")
    print(f"  Periodic Scanners (32-47) | Pulsar Mean: {np.mean(A0_pulsar[:, 32:48]):.4f} | Noise Mean: {np.mean(A0_noise[:, 32:48]):.4f}")
    print(f"  Lorentzian Extract (48-63)| Pulsar Mean: {np.mean(A0_pulsar[:, 48:64]):.4f} | Noise Mean: {np.mean(A0_noise[:, 48:64]):.4f}")

    # Layer 1 Council Analysis (with tanh-approximated silu/swish)
    groups = ["Squasher (tanh)", "AND Gate (Swish-tanh)", "Aggregator (softsign)", "XOR Gate (gaussian)"]
    pulsar_means = [
        np.mean(A1_pulsar[:, 0:4]), 
        np.mean(A1_pulsar[:, 4:8]), 
        np.mean(A1_pulsar[:, 8:12]), 
        np.mean(A1_pulsar[:, 12:16])
    ]
    noise_means = [
        np.mean(A1_noise[:, 0:4]), 
        np.mean(A1_noise[:, 4:8]), 
        np.mean(A1_noise[:, 8:12]), 
        np.mean(A1_noise[:, 12:16])
    ]
    
    plt.figure(figsize=(10, 5))
    x_coords = np.arange(len(groups))
    plt.bar(x_coords - 0.2, pulsar_means, 0.4, label='Pulsars Mean Output', color='tab:blue')
    plt.bar(x_coords + 0.2, noise_means, 0.4, label='RFI Noise Mean Output', color='tab:red')
    plt.xticks(x_coords, groups)
    plt.title("Layer 1: Council Activation States (Pulsar vs. Noise RFI)")
    plt.ylabel("Mean Activation Value")
    plt.legend()
    plt.grid(True, axis='y', ls='--', alpha=0.3)
    plt.show()

analyze_and_plot_fourier_frequencies(hetero_model)
calculate_and_plot_activation_statistics(X_bridge, y_true, hetero_model, eval_agent)

## 9. Plotting Rescued Pulsars

We identify candidate pulsar signals that were *missed* by the strict model ($v=1.75$) but successfully *rescued* by the sensitive model ($v=0.65$). We load and plot their raw 2D DM-SNR and Phase-Time surfaces to inspect what they look like.

In [ ]:
def isolate_and_plot_pulsars(X_bridge, y_true, model, test_loader):
    # Predictions for strict threshold (v=1.75)
    model.change_constants({2: [(0, "v", 1.75)]})
    agent = Wrapper(model, work_type="class")
    preds_strict = (agent.predict(X_bridge) > 0.5).astype(int).flatten()
    
    # Predictions for sensitive threshold (v=0.65)
    model.change_constants({2: [(0, "v", 0.65)]})
    preds_sensitive = (agent.predict(X_bridge) > 0.5).astype(int).flatten()
    
    # Rescued stars: true pulsars (label=0) that strict missed (pred=1) but sensitive found (pred=0)
    exclusive_mask = (y_true == 0) & (preds_strict == 1) & (preds_sensitive == 0)
    exclusive_indices = np.where(exclusive_mask)[0]
    print(f"Found {len(exclusive_indices)} exclusive pulsars rescued by the tuned math!")
    
    if len(exclusive_indices) == 0:
        return
        
    # Load raw images from DataLoader
    all_images = []
    for images, _ in test_loader:
        all_images.append(images.numpy())
    all_images = np.vstack(all_images)
    
    # Plot the first 2 rescued candidates
    num_plot = min(2, len(exclusive_indices))
    fig, axes = plt.subplots(num_plot, 2, figsize=(10, 4 * num_plot))
    if num_plot == 1:
        axes = [axes]
        
    for i in range(num_plot):
        idx = exclusive_indices[i]
        img_data = all_images[idx]
        dm_surface = img_data[0]   # Channel 0: DM-SNR curve
        time_surface = img_data[2] # Channel 2: Phase-Time curve
        
        axes[i][0].imshow(dm_surface, aspect='auto', cmap='plasma')
        axes[i][0].set_title(f"Pulsar Index {idx} | DM-SNR Surface")
        axes[i][0].set_ylabel("Dispersion Measure")
        axes[i][0].set_xlabel("Period Correction")
        
        axes[i][1].imshow(time_surface, aspect='auto', cmap='viridis')
        axes[i][1].set_title(f"Pulsar Index {idx} | Phase-Time Surface")
        axes[i][1].set_ylabel("Sub-integration Time")
        axes[i][1].set_xlabel("Phase")
        
    plt.tight_layout()
    plt.show()

isolate_and_plot_pulsars(X_bridge, y_true, hetero_model, test_loader)

## 10. Sensitivity Threshold Sweep Analysis

We sweep the output constant parameter `v` across 50 values between $0.1$ and $5.0$. For each step, we evaluate performance and plot the resulting trade-off curves (Found Stars vs. False Alarms vs. Missed Stars) to discover the optimal sensitivity threshold.

In [ ]:
def plot_dial_sweep(model, agent, X, y, v_min=0.1, v_max=5.0, steps=50):
    v_values = np.linspace(v_min, v_max, steps)
    found_stars = []
    missed_stars = []
    false_alarms = []
    
    for v in v_values:
        # Change output constant and evaluate
        model.change_constants({2: [(0, "v", v)]})
        _, counts = agent.test_accuracy(X, y)
        
        found_stars.append(counts['correc_pos'])
        missed_stars.append(counts['false_neg'])
        false_alarms.append(counts['false_pos'])
        
    plt.figure(figsize=(12, 6))
    plt.plot(v_values, found_stars, label='Found Stars (True Positives)', color='lime', linewidth=3)
    plt.plot(v_values, false_alarms, label='False Alarms (RFI Noise)', color='red', linewidth=3)
    plt.plot(v_values, missed_stars, label='Missed Stars (False Negatives)', color='orange', linestyle='--', linewidth=2)
    
    plt.axvline(x=1.0, color='gray', linestyle=':', label='Standard v=1.0')
    plt.title("Zero-Recompile Sensitivity Dial Sweep", fontsize=16, fontweight='bold')
    plt.xlabel("Output Sensitivity Param 'v'", fontsize=14)
    plt.ylabel("Sample Count", fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.show()

plot_dial_sweep(hetero_model, eval_agent, X_bridge, y_true, v_min=0.1, v_max=5.0, steps=50)